In [1]:
import pandas as pd
import numpy as np

Kalenderbaserte features

In [3]:
df = pd.read_csv(
    "../data/consumption_selected.csv",
    sep=";",
    decimal=","
)

# Kalenderbaserte features
df["HOUR_SIN"] = np.sin(2 * np.pi * df["HOUR"] / 24)
df["HOUR_COS"] = np.cos(2 * np.pi * df["HOUR"] / 24)

df["MONTH_SIN"] = np.sin(2 * np.pi * (df["MONTH"] - 1) / 12)
df["MONTH_COS"] = np.cos(2 * np.pi * (df["MONTH"] - 1) / 12)

df["DAY_SIN"] = np.sin(2 * np.pi * df["DAY_OF_WEEK"] / 7)
df["DAY_COS"] = np.cos(2 * np.pi * df["DAY_OF_WEEK"] / 7)

df["IS_WEEKEND"] = (df["DAY_OF_WEEK"] >= 5).astype(int)

calendar_features = [
    "HOUR_SIN", "HOUR_COS",
    "MONTH_SIN", "MONTH_COS",
    "DAY_SIN", "DAY_COS",
    "IS_WEEKEND",
    "KOMMUNE",
    "FORBRUKSGRUPPE"
]

print(df[calendar_features].head())

   HOUR_SIN  HOUR_COS  MONTH_SIN  MONTH_COS   DAY_SIN  DAY_COS  IS_WEEKEND  \
0       0.0       1.0        0.5   0.866025  0.781831  0.62349           0   
1       0.0       1.0        0.5   0.866025  0.781831  0.62349           0   
2       0.0       1.0        0.5   0.866025  0.781831  0.62349           0   
3       0.0       1.0        0.5   0.866025  0.781831  0.62349           0   
4       0.0       1.0        0.5   0.866025  0.781831  0.62349           0   

  KOMMUNE FORBRUKSGRUPPE  
0    Alta     Forretning  
1    Alta       Industri  
2    Alta         Privat  
3  Bergen     Forretning  
4  Bergen       Industri  


Historiske prediktorene

In [4]:
# Parse tidspunktene til norsk lokal tid
df["TIMESTAMP"] = (
    pd.to_datetime(df["STARTTID"], utc=True)
      .dt.tz_convert("Europe/Oslo")
)

df["DATE"] = df["TIMESTAMP"].dt.date
df["LOCAL_HOUR"] = df["TIMESTAMP"].dt.hour

keys = ["KOMMUNE", "FORBRUKSGRUPPE"]

# Daglige gjennomsnitt
daily = (
    df.groupby(keys + ["DATE"])["VOLUM_KWH"]
      .mean()
      .reset_index(name="DAILY_MEAN")
)

# Lag hjelpekolonne med dato
df["DATE_KEY"] = (
    df["TIMESTAMP"]
    .dt.tz_localize(None)
    .dt.normalize()
)

daily["DATE_KEY"] = pd.to_datetime(daily["DATE"])

# Gjennomsnitt to kalenderdøgn tidligere
previous = daily[keys + ["DATE_KEY", "DAILY_MEAN"]].copy()
previous["DATE_KEY"] += pd.Timedelta(days=2)
previous = previous.rename(
    columns={"DAILY_MEAN": "PREV_DAY_MEAN"}
)

df = df.merge(
    previous,
    on=keys + ["DATE_KEY"],
    how="left",
    validate="many_to_one"
)

# Timeforbruk 2 og 7 kalenderdøgn tidligere
for days in [2, 7]:
    history = df[
        keys + ["DATE_KEY", "LOCAL_HOUR", "VOLUM_KWH"]
    ].copy()

    history["DATE_KEY"] += pd.Timedelta(days=days)
    history = history.rename(
        columns={"VOLUM_KWH": f"LAG_{days}D"}
    )

    # Unngå mange-til-mange-merge ved gjentatt time
    # under overgang til vintertid
    history = (
        history.groupby(
            keys + ["DATE_KEY", "LOCAL_HOUR"],
            as_index=False
        )[f"LAG_{days}D"].mean()
    )

    df = df.merge(
        history,
        on=keys + ["DATE_KEY", "LOCAL_HOUR"],
        how="left",
        validate="many_to_one"
    )

print(
    df[[
        "KOMMUNE", "FORBRUKSGRUPPE",
        "DATE_KEY", "LOCAL_HOUR",
        "VOLUM_KWH", "LAG_2D",
        "LAG_7D", "PREV_DAY_MEAN"
    ]].tail(10)
)

           KOMMUNE FORBRUKSGRUPPE   DATE_KEY  LOCAL_HOUR   VOLUM_KWH  \
1115960    Sunndal         Privat 2026-04-30          23    5328.694   
1115961     Tromsø     Forretning 2026-04-30          23   40554.563   
1115962     Tromsø       Industri 2026-04-30          23    9426.388   
1115963     Tromsø         Privat 2026-04-30          23   80151.881   
1115964  Trondheim     Forretning 2026-04-30          23   74044.931   
1115965  Trondheim       Industri 2026-04-30          23   38626.449   
1115966  Trondheim         Privat 2026-04-30          23  144997.318   
1115967      Årdal     Forretning 2026-04-30          23    1619.537   
1115968      Årdal       Industri 2026-04-30          23  390816.464   
1115969      Årdal         Privat 2026-04-30          23    4130.784   

             LAG_2D      LAG_7D  PREV_DAY_MEAN  
1115960    6194.545    6372.506    6024.442542  
1115961   44391.189   49477.595   55348.055250  
1115962   10542.804   10054.141   12342.449875  
1115963   8

Kontroll

In [5]:
lag_features = [
    "LAG_2D",
    "LAG_7D",
    "PREV_DAY_MEAN"
]

print("Antall observasjoner:", len(df))

print("\nManglende historiske verdier:")
print(df[lag_features].isna().sum())

print("\nAndel manglende (%):")
print(
    (df[lag_features].isna().mean() * 100).round(2)
)

print("\nDuplikater per kommune, gruppe og tidspunkt:")
print(
    df.duplicated(
        subset=["KOMMUNE", "FORBRUKSGRUPPE", "TIMESTAMP"]
    ).sum()
)

Antall observasjoner: 1115970

Manglende historiske verdier:
LAG_2D           1590
LAG_7D           5190
PREV_DAY_MEAN    1440
dtype: int64

Andel manglende (%):
LAG_2D           0.14
LAG_7D           0.47
PREV_DAY_MEAN    0.13
dtype: float64

Duplikater per kommune, gruppe og tidspunkt:
0


Kontroll

In [6]:
lag_features = [
    "LAG_2D",
    "LAG_7D",
    "PREV_DAY_MEAN"
]

# Finn observasjoner som mangler minst én lag feature
missing = df[df[lag_features].isna().any(axis=1)]

print("Manglende observasjoner per år:")
print(missing.groupby("YEAR").size())

print("\nManglende observasjoner per kommune:")
print(
    missing.groupby("KOMMUNE")
    .size()
    .sort_values(ascending=False)
)

print("\nAntall komplette observasjoner:")
print(df.dropna(subset=lag_features).shape[0])

Manglende observasjoner per år:
YEAR
2022    5100
2023      60
2024      60
2025      60
2026      60
dtype: int64

Manglende observasjoner per kommune:
KOMMUNE
Alta            534
Bergen          534
Bodø            534
Kristiansand    534
Oslo            534
Stavanger       534
Sunndal         534
Tromsø          534
Trondheim       534
Årdal           534
dtype: int64

Antall komplette observasjoner:
1110630


Kontroll

In [7]:
missing_dates = (
    df.loc[df[lag_features].isna().any(axis=1)]
      .groupby("DATE_KEY")
      .size()
)

print(missing_dates.to_string())


DATE_KEY
2022-02-01    720
2022-02-02    720
2022-02-03    720
2022-02-04    720
2022-02-05    720
2022-02-06    720
2022-02-07    720
2022-03-29     30
2022-04-03     30
2023-03-28     30
2023-04-02     30
2024-04-02     30
2024-04-07     30
2025-04-01     30
2025-04-06     30
2026-03-31     30
2026-04-05     30


Fjerner observasjoner med manglende historikk

In [8]:
# Fjern observasjoner med manglende historiske prediktorer
df_model = df.dropna(subset=lag_features).copy()

# Kontroller resultatet
print("Før:", len(df))
print("Etter:", len(df_model))
print("Fjernet:", len(df) - len(df_model))

print("\nManglende historiske verdier:")
print(df_model[lag_features].isna().sum())

Før: 1115970
Etter: 1110630
Fjernet: 5340

Manglende historiske verdier:
LAG_2D           0
LAG_7D           0
PREV_DAY_MEAN    0
dtype: int64


Train / Validation / Test Split

In [9]:
# Train: 2022–2024
train = df_model[df_model["YEAR"] <= 2024].copy()

# Validation: 2025
validation = df_model[df_model["YEAR"] == 2025].copy()

# Test: 2026
test = df_model[df_model["YEAR"] == 2026].copy()

# Kontroller oppdelingen
for name, subset in [
    ("Train", train),
    ("Validation", validation),
    ("Test", test)
]:
    print(
        f"{name}: {len(subset):,} observasjoner | "
        f"{subset['DATE_KEY'].min().date()} "
        f"til {subset['DATE_KEY'].max().date()}"
    )

print("\nTotalt:", len(train) + len(validation) + len(test))

Train: 761,580 observasjoner | 2022-02-08 til 2024-12-31
Validation: 262,740 observasjoner | 2025-01-01 til 2025-12-31
Test: 86,310 observasjoner | 2026-01-01 til 2026-04-30

Totalt: 1110630


In [10]:
# Modell A: Kun kalenderbaserte prediktorer
features_A = calendar_features.copy()

# Modell B: Kalender + historisk strømforbruk
features_B = calendar_features + lag_features

target = "VOLUM_KWH"

print("Modell A:", features_A)
print("\nModell B:", features_B)

Modell A: ['HOUR_SIN', 'HOUR_COS', 'MONTH_SIN', 'MONTH_COS', 'DAY_SIN', 'DAY_COS', 'IS_WEEKEND', 'KOMMUNE', 'FORBRUKSGRUPPE']

Modell B: ['HOUR_SIN', 'HOUR_COS', 'MONTH_SIN', 'MONTH_COS', 'DAY_SIN', 'DAY_COS', 'IS_WEEKEND', 'KOMMUNE', 'FORBRUKSGRUPPE', 'LAG_2D', 'LAG_7D', 'PREV_DAY_MEAN']


In [11]:
# Modell A
X_train_A = train[features_A].copy()
X_val_A = validation[features_A].copy()
X_test_A = test[features_A].copy()

# Modell B
X_train_B = train[features_B].copy()
X_val_B = validation[features_B].copy()
X_test_B = test[features_B].copy()

# Samme target for begge modellene
y_train = train[target].copy()
y_val = validation[target].copy()
y_test = test[target].copy()

print("Modell A:", X_train_A.shape)
print("Modell B:", X_train_B.shape)
print("Target:", y_train.shape)

Modell A: (761580, 9)
Modell B: (761580, 12)
Target: (761580,)


Preprocessing

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Kategoriske prediktorer
categorical_features = [
    "KOMMUNE",
    "FORBRUKSGRUPPE"
]

# Numeriske prediktorer for hver modell
numerical_A = [
    col for col in features_A
    if col not in categorical_features
]

numerical_B = [
    col for col in features_B
    if col not in categorical_features
]

# Preprocessing for Modell A
preprocessor_A = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_A),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features)
    ]
)

# Preprocessing for Modell B
preprocessor_B = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_B),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features)
    ]
)

print("Numeriske variabler A:", numerical_A)
print("Numeriske variabler B:", numerical_B)
print("\nPreprocessing opprettet.")

Numeriske variabler A: ['HOUR_SIN', 'HOUR_COS', 'MONTH_SIN', 'MONTH_COS', 'DAY_SIN', 'DAY_COS', 'IS_WEEKEND']
Numeriske variabler B: ['HOUR_SIN', 'HOUR_COS', 'MONTH_SIN', 'MONTH_COS', 'DAY_SIN', 'DAY_COS', 'IS_WEEKEND', 'LAG_2D', 'LAG_7D', 'PREV_DAY_MEAN']

Preprocessing opprettet.


Baseline A

In [13]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Gjennomsnitt fra treningsdataene
baseline_A = (
    train.groupby([
        "KOMMUNE",
        "FORBRUKSGRUPPE",
        "DAY_OF_WEEK",
        "HOUR"
    ])["VOLUM_KWH"]
    .mean()
    .rename("PREDICTION")
    .reset_index()
)

# Koble prediksjonene til valideringsdataene
val_baseline_A = validation.merge(
    baseline_A,
    on=["KOMMUNE", "FORBRUKSGRUPPE", "DAY_OF_WEEK", "HOUR"],
    how="left",
    validate="many_to_one"
)

# Kontroller at alle observasjoner fikk prediksjon
print("Manglende prediksjoner:",
      val_baseline_A["PREDICTION"].isna().sum())

# Evaluer
y_pred_A = val_baseline_A["PREDICTION"]

mae_A = mean_absolute_error(y_val, y_pred_A)
rmse_A = np.sqrt(mean_squared_error(y_val, y_pred_A))

print(f"Baseline A - MAE: {mae_A:,.2f} kWh")
print(f"Baseline A - RMSE: {rmse_A:,.2f} kWh")

Manglende prediksjoner: 0
Baseline A - MAE: 20,365.13 kWh
Baseline A - RMSE: 40,934.72 kWh


Baseline B

In [14]:
# Prediksjon = forbruk samme time uken før
y_pred_B = validation["LAG_7D"]

mae_B = mean_absolute_error(y_val, y_pred_B)
rmse_B = np.sqrt(mean_squared_error(y_val, y_pred_B))

print(f"Baseline B - MAE: {mae_B:,.2f} kWh")
print(f"Baseline B - RMSE: {rmse_B:,.2f} kWh")

Baseline B - MAE: 9,248.14 kWh
Baseline B - RMSE: 21,594.34 kWh


Opprette modell

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# Modell A: Kalenderbaserte prediktorer
model_A = Pipeline([
    ("preprocessing", preprocessor_A),
    ("regression", LinearRegression())
])

# Modell B: Kalender + historisk forbruk
model_B = Pipeline([
    ("preprocessing", preprocessor_B),
    ("regression", LinearRegression())
])

print("Begge regresjonsmodellene er opprettet.")

Begge regresjonsmodellene er opprettet.


Trainig

In [ ]:
# Tren Modell A
model_A.fit(X_train_A, y_train)

# Tren Modell B
model_B.fit(X_train_B, y_train)

print("Begge modellene er ferdig trent.")

Begge modellene er ferdig trent.


Evaluering på valideringsdata

In [18]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np
import pandas as pd

# Prediksjoner på valideringsdata
pred_A = model_A.predict(X_val_A)
pred_B = model_B.predict(X_val_B)

# Sammenlign modellene
results = pd.DataFrame({
    "Model": ["Linear Regression A", "Linear Regression B"],
    "MAE": [
        mean_absolute_error(y_val, pred_A),
        mean_absolute_error(y_val, pred_B)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_val, pred_A)),
        np.sqrt(mean_squared_error(y_val, pred_B))
    ],
    "R2": [
        r2_score(y_val, pred_A),
        r2_score(y_val, pred_B)
    ]
})

print(results.round(3).to_string(index=False))

              Model       MAE       RMSE    R2
Linear Regression A 89738.787 133348.066 0.339
Linear Regression B  8492.746  17740.566 0.988


Undersøkelse av den store diffen

In [19]:
# Sammenlign lineær regresjon B med historisk baseline

comparison = pd.DataFrame({
    "Model": [
        "Baseline B (LAG_7D)",
        "Linear Regression B"
    ],
    "MAE": [
        mean_absolute_error(y_val, validation["LAG_7D"]),
        mean_absolute_error(y_val, pred_B)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_val, validation["LAG_7D"])),
        np.sqrt(mean_squared_error(y_val, pred_B))
    ],
    "R2": [
        r2_score(y_val, validation["LAG_7D"]),
        r2_score(y_val, pred_B)
    ]
})

print(comparison.round(3).to_string(index=False))


              Model      MAE      RMSE    R2
Baseline B (LAG_7D) 9248.137 21594.339 0.983
Linear Regression B 8492.746 17740.566 0.988


Evaluering av modellene per forbruksgruppe

In [20]:
# Lag evalueringsdatasett
evaluation = validation[
    ["KOMMUNE", "FORBRUKSGRUPPE", "VOLUM_KWH", "LAG_7D"]
].copy()

evaluation["PRED_A"] = pred_A
evaluation["PRED_B"] = pred_B

# Evaluer per forbruksgruppe
group_results = []

for group, subset in evaluation.groupby("FORBRUKSGRUPPE"):
    actual = subset["VOLUM_KWH"]

    for model_name, prediction in [
        ("Baseline B", subset["LAG_7D"]),
        ("Linear Regression A", subset["PRED_A"]),
        ("Linear Regression B", subset["PRED_B"])
    ]:
        group_results.append({
            "Group": group,
            "Model": model_name,
            "MAE": mean_absolute_error(actual, prediction),
            "RMSE": np.sqrt(mean_squared_error(actual, prediction)),
            "R2": r2_score(actual, prediction)
        })

group_results = pd.DataFrame(group_results)

print(
    group_results
    .sort_values(["Group", "MAE"])
    .round(3)
    .to_string(index=False)
)


     Group               Model        MAE       RMSE    R2
Forretning          Baseline B   6434.483  14639.384 0.984
Forretning Linear Regression B   7737.454  16127.938 0.981
Forretning Linear Regression A  61145.545  89178.904 0.420
  Industri Linear Regression B   7053.453  16586.028 0.994
  Industri          Baseline B   7839.892  20876.715 0.990
  Industri Linear Regression A 131886.875 180981.797 0.281
    Privat Linear Regression B  10687.330  20223.163 0.979
    Privat          Baseline B  13470.035  27364.168 0.962
    Privat Linear Regression A  76183.940 112418.112 0.362


Ridge Regression

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

# Ridge-modell med samme preprocessing som tidligere
ridge_B = Pipeline([
    ("preprocessing", preprocessor_B),
    ("regression", Ridge(alpha=1.0))
])

# Tren på 2022–2024
ridge_B.fit(X_train_B, y_train)

# Prediker 2025
pred_ridge_B = ridge_B.predict(X_val_B)

# Evaluer
print("Ridge B")
print(
    "MAE:",
    round(mean_absolute_error(y_val, pred_ridge_B), 3)
)
print(
    "RMSE:",
    round(np.sqrt(mean_squared_error(y_val, pred_ridge_B)), 3)
)
print(
    "R2:",
    round(r2_score(y_val, pred_ridge_B), 3)
)

Ridge B
MAE: 8492.784
RMSE: 17740.551
R2: 0.988


Random Forest

In [22]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

# Preprocessing uten skalering
rf_preprocessor_B = ColumnTransformer([
    ("num", "passthrough", numerical_B),
    ("cat", OneHotEncoder(handle_unknown="ignore"),
     categorical_features)
])

# Random Forest
rf_B = Pipeline([
    ("preprocessing", rf_preprocessor_B),
    ("regression", RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_leaf=5,
        max_features=1.0,
        random_state=42,
        n_jobs=-1
    ))
])

print("Random Forest B opprettet.")

Random Forest B opprettet.


Trening og evaluring

In [23]:
# Tren modellen på 2022–2024
rf_B.fit(X_train_B, y_train)

# Prediker på 2025
pred_rf_B = rf_B.predict(X_val_B)

# Evaluer
print("Random Forest B")
print("MAE:", round(mean_absolute_error(y_val, pred_rf_B), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(y_val, pred_rf_B)), 3))
print("R2:", round(r2_score(y_val, pred_rf_B), 3))

Random Forest B
MAE: 6466.854
RMSE: 14691.663
R2: 0.992


Evaluer Random Forest per forbruksgruppe

In [24]:
# Legg til Random Forest-prediksjonene
evaluation["PRED_RF_B"] = pred_rf_B

# Sammenlign modellene per forbruksgruppe
rf_group_results = []

for group, subset in evaluation.groupby("FORBRUKSGRUPPE"):
    actual = subset["VOLUM_KWH"]

    for model_name, prediction in [
        ("Baseline B", subset["LAG_7D"]),
        ("Linear Regression B", subset["PRED_B"]),
        ("Random Forest B", subset["PRED_RF_B"])
    ]:
        rf_group_results.append({
            "Group": group,
            "Model": model_name,
            "MAE": mean_absolute_error(actual, prediction),
            "RMSE": np.sqrt(mean_squared_error(actual, prediction)),
            "R2": r2_score(actual, prediction)
        })

rf_group_results = pd.DataFrame(rf_group_results)

print(
    rf_group_results
    .sort_values(["Group", "MAE"])
    .round(3)
    .to_string(index=False)
)

     Group               Model       MAE      RMSE    R2
Forretning     Random Forest B  4658.486 10535.540 0.992
Forretning          Baseline B  6434.483 14639.384 0.984
Forretning Linear Regression B  7737.454 16127.938 0.981
  Industri     Random Forest B  6085.915 15489.904 0.995
  Industri Linear Regression B  7053.453 16586.028 0.994
  Industri          Baseline B  7839.892 20876.715 0.990
    Privat     Random Forest B  8656.163 17222.083 0.985
    Privat Linear Regression B 10687.330 20223.163 0.979
    Privat          Baseline B 13470.035 27364.168 0.962


Random Forest A

In [25]:
# Preprocessing for Random Forest A
rf_preprocessor_A = ColumnTransformer([
    ("num", "passthrough", numerical_A),
    ("cat", OneHotEncoder(handle_unknown="ignore"),
     categorical_features)
])

# Random Forest A
rf_A = Pipeline([
    ("preprocessing", rf_preprocessor_A),
    ("regression", RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_leaf=5,
        max_features=1.0,
        random_state=42,
        n_jobs=-1
    ))
])

# Tren modellen
rf_A.fit(X_train_A, y_train)

print("Random Forest A ferdig trent.")


Random Forest A ferdig trent.


Evaluering og validering A

In [26]:
# Prediksjon på valideringsdataene
pred_rf_A = rf_A.predict(X_val_A)

comparison_rf = pd.DataFrame({
    "Model": [
        "Linear Regression A",
        "Random Forest A",
        "Linear Regression B",
        "Random Forest B"
    ],
    "MAE": [
        mean_absolute_error(y_val, pred_A),
        mean_absolute_error(y_val, pred_rf_A),
        mean_absolute_error(y_val, pred_B),
        mean_absolute_error(y_val, pred_rf_B)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_val, pred_A)),
        np.sqrt(mean_squared_error(y_val, pred_rf_A)),
        np.sqrt(mean_squared_error(y_val, pred_B)),
        np.sqrt(mean_squared_error(y_val, pred_rf_B))
    ],
    "R2": [
        r2_score(y_val, pred_A),
        r2_score(y_val, pred_rf_A),
        r2_score(y_val, pred_B),
        r2_score(y_val, pred_rf_B)
    ]
})

print(comparison_rf.round(3).to_string(index=False))


              Model       MAE       RMSE    R2
Linear Regression A 89738.787 133348.066 0.339
    Random Forest A  9384.412  18852.852 0.987
Linear Regression B  8492.746  17740.566 0.988
    Random Forest B  6466.854  14691.663 0.992


Evaluering av Random Forest A og B per kommune

In [27]:
# Opprett evalueringsdata
municipality_eval = validation[
    ["KOMMUNE", "FORBRUKSGRUPPE", "VOLUM_KWH"]
].copy()

municipality_eval["PRED_RF_A"] = pred_rf_A
municipality_eval["PRED_RF_B"] = pred_rf_B

# Beregn MAE per kommune
municipality_results = []

for municipality, subset in municipality_eval.groupby("KOMMUNE"):
    actual = subset["VOLUM_KWH"]

    mae_A = mean_absolute_error(
        actual, subset["PRED_RF_A"]
    )

    mae_B = mean_absolute_error(
        actual, subset["PRED_RF_B"]
    )

    municipality_results.append({
        "Kommune": municipality,
        "MAE_A": mae_A,
        "MAE_B": mae_B,
        "Improvement_%": (mae_A - mae_B) / mae_A * 100
    })

municipality_results = pd.DataFrame(municipality_results)

print(
    municipality_results
    .sort_values("Improvement_%", ascending=False)
    .round(2)
    .to_string(index=False)
)


     Kommune    MAE_A    MAE_B  Improvement_%
       Årdal  3189.51  1178.79          63.04
     Sunndal  5009.06  2459.95          50.89
Kristiansand  6896.05  3960.54          42.57
   Stavanger  7349.35  4678.43          36.34
   Trondheim 13211.22  9335.16          29.34
      Bergen 13049.74  9392.82          28.02
      Tromsø  5218.03  3776.78          27.62
        Oslo 34368.28 25716.67          25.17
        Bodø  3333.26  2496.38          25.11
        Alta  2219.63  1673.01          24.63


Gradient Boosting B